In [1]:
import wrds
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

db = wrds.Connection()

FVOL_COLS = ['actq','rectq','invtq','ppentq','atq','lctq','dlcq',
             'apq','dlttq','ltq','seqq','xoprq','cogsq','xsgaq']

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [2]:
import os

START_DATE = '2020-01-01'

if os.path.exists('comp_fundq_raw.parquet'):
    comp_raw = pd.read_parquet('comp_fundq_raw.parquet')
    print('loaded from cache:', comp_raw.shape)
else:
    select_cols = (['gvkey','datadate','rdq','fyearq','fqtr','saleq']
                   + FVOL_COLS
                   + ['revtq','xintq','ibq','txditcq','pstkrq','pstkq'])
    col_str = ', '.join(select_cols)
    query = f"""
        SELECT {col_str}
        FROM comp.fundq
        WHERE indfmt='INDL' AND datafmt='STD' AND consol='C' AND popsrc='D'
          AND datadate >= '{START_DATE}'
    """
    comp_raw = db.raw_sql(query, date_cols=['datadate','rdq'])
    comp_raw = comp_raw.drop_duplicates(subset=['gvkey','datadate'])
    comp_raw = comp_raw.sort_values(['gvkey','datadate']).reset_index(drop=True)
    comp_raw.to_parquet('comp_fundq_raw.parquet')
    print('extracted from WRDS:', comp_raw.shape)

extracted from WRDS: (312694, 26)


In [3]:
def build_fvol(input_file, fvol_cols=FVOL_COLS,
               min_items=10, roll_window=8, roll_min=4, min_rank_items=10):

    df = pd.read_parquet(input_file)
    df = df.drop_duplicates(subset=['gvkey','datadate'])
    df = df.sort_values(['gvkey','datadate']).reset_index(drop=True)

    df['n_valid'] = df[fvol_cols].notna().sum(axis=1)
    df = df[df['n_valid'] >= min_items].reset_index(drop=True)

    g = df.groupby('gvkey', sort=False)
    for c in fvol_cols:
        df[f'd_{c}'] = df[c] - g[c].shift(4)

    g = df.groupby('gvkey', sort=False)
    for c in fvol_cols:
        df[f'std_{c}'] = (g[f'd_{c}']
                          .rolling(window=roll_window, min_periods=roll_min)
                          .std().reset_index(level=0, drop=True))

    df['avg_sales'] = (g['saleq']
                       .rolling(window=roll_window, min_periods=roll_min)
                       .mean().reset_index(level=0, drop=True))
    df['avg_sales_valid'] = df['avg_sales'].where(df['avg_sales'] > 0)

    fvol_individual = []
    for c in fvol_cols:
        df[f'fvol_{c}'] = df[f'std_{c}'] / df['avg_sales_valid']
        fvol_individual.append(f'fvol_{c}')

    ranked = df.groupby('datadate')[fvol_individual].rank(pct=True)
    ranked.columns = [f'rank_{c}' for c in fvol_cols]
    df = pd.concat([df, ranked], axis=1)

    rank_cols = [f'rank_{c}' for c in fvol_cols]
    df['n_valid_rank'] = df[rank_cols].notna().sum(axis=1)
    df['FVOL'] = df[rank_cols].mean(axis=1)
    df.loc[df['n_valid_rank'] < min_rank_items, 'FVOL'] = np.nan

    return df

df = build_fvol('comp_fundq_raw.parquet')
df.to_parquet('comp_with_fvol.parquet')

In [4]:
print('rows:', len(df))
print('FVOL non-null:', df['FVOL'].notna().sum())
print(df['FVOL'].describe().round(4))
print('avg valid firms/qtr:',
      round(df[df['FVOL'].notna()].groupby('datadate')['gvkey'].nunique().mean(), 0))
print(df['FVOL'].quantile([0, .01, .25, .5, .75, .99, 1]).round(4))

rows: 183210
FVOL non-null: 102840
count    102840.0000
mean          0.5029
std           0.1999
min           0.0344
25%           0.3463
50%           0.4894
75%           0.6468
max           0.9996
Name: FVOL, dtype: float64
avg valid firms/qtr: 1836.0
0.00    0.0344
0.01    0.1277
0.25    0.3463
0.50    0.4894
0.75    0.6468
0.99    0.9475
1.00    0.9996
Name: FVOL, dtype: float64
